# _template: Notebook Template

**Session:** (none — this is the template every notebook starts from)
**Expected runtime:** 2 minutes on Colab free-tier CPU or local
**Needs:** nothing — no API key, no GPU, no data files
**A correct result looks like:** the final cell prints `TEMPLATE OK`
with the environment name and a checkpoint round-trip confirmed.

> All data in this lab is synthetic. No real OQ material anywhere.

---
*How to use this file: copy it, rename it `NN_lab_name.ipynb`, fill in
the header above, and build the lab from Cell 4 onward. The full rules
are in `docs/notebook_conventions.md`.*

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** each notebook installs only what it needs, with
exact pins matching `requirements.txt`. Colab already has most of the
stack preinstalled, and local machines installed `requirements.txt`
during setup — so this cell is small and fast.

In [ ]:
# Pinned installs — versions match requirements.txt. Colab only.
if IN_COLAB:
    %pip install -q requests==2.32.4 python-dotenv==1.2.3
print("Install cell done.")

**Why this cell:** every lab that talks to a model goes through
`config/endpoints.py` (Interface Contract #3). Switching between the
local Ollama model, the hosted API and the tuned adapter is one
string — the lab code never changes. This cell just shows what is
configured; it makes no network calls.

In [ ]:
import utils  # noqa: F401  (shared helpers from notebooks/utils.py)
from config.endpoints import list_endpoints

endpoints = list_endpoints()
for name, description in endpoints.items():
    print(f"{name:<7} -> {description}")

# In a real lab you would now pick one:
#   from config.endpoints import get_endpoint
#   llm = get_endpoint("hosted")
#   reply = llm.chat("...")

**How TODO gaps look** (participant notebooks only — this template runs
clean, so the example below is shown, not executed):

```python
# ── TODO 1 ─────────────────────────────────────────────────────────
# Build the prompt that asks the model for STRICT JSON matching the
# ticket schema. Hint: tell it what to do with missing fields.
prompt = ...  # <- replace the ... with your code
# ───────────────────────────────────────────────────────────────
```

The `...` makes the unmodified cell fail loudly instead of silently
doing nothing. The matching notebook in `solutions/` fills every gap,
same numbering.

**Why this cell:** runtimes disconnect. Every milestone saves to
`CHECKPOINT_DIR` (which is on Google Drive in Colab), and every
milestone cell loads its checkpoint if one exists instead of
recomputing. A disconnect costs the current cell, never the session.
This cell demonstrates the round trip.

In [ ]:
# Milestone pattern: load if the checkpoint exists, compute if not.
demo_result = utils.load_json(CHECKPOINT_DIR, "_template_demo", default=None)

if demo_result is None:
    # Stand-in for real work (an ingestion, a training run, an eval).
    demo_result = {"answer": 42, "rows_processed": 3}
    utils.save_json(CHECKPOINT_DIR, "_template_demo", demo_result)
    came_from = "fresh run"
else:
    came_from = "checkpoint"

print(f"demo_result = {demo_result}  (from {came_from})")

**Why this cell:** the last cell always prints the result the header
promised, so "done" is checkable at a glance.

In [ ]:
# The header promised: `TEMPLATE OK` + environment + checkpoint confirmation.
roundtrip = utils.load_json(CHECKPOINT_DIR, "_template_demo")
assert roundtrip["answer"] == 42, "checkpoint round-trip failed"

print(f"TEMPLATE OK — environment={'Colab' if IN_COLAB else 'local'}, "
      f"checkpoint round-trip confirmed: {roundtrip}")